In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [4]:
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
       nn.Linear(28*28, 512),
       nn.ReLU(),
       nn.Linear(512, 512),
       nn.ReLU(),
       nn.Linear(512, 10),
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

In [5]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [6]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted Class: {y_pred}")

Predicted Class: tensor([6])


Model Layers

In [7]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [9]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


nn.Linear

In [10]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU

In [11]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.4092, -0.1390, -0.1001,  0.9743, -0.3377,  0.0246, -0.2585,  0.0629,
         -0.5684, -0.5861, -0.3108,  0.2805,  0.1764,  0.0338, -0.4238,  0.1114,
          0.0753,  0.0130,  0.0643,  0.4758],
        [ 0.2024, -0.4042, -0.3295,  0.4911, -0.3951,  0.0904, -0.1465, -0.4478,
         -0.1173, -0.3879, -0.2849, -0.0270,  0.2337,  0.2247, -0.3739, -0.1845,
          0.5123,  0.2381,  0.0793,  0.5395],
        [ 0.2620, -0.1097, -0.1325,  0.4978, -0.2229,  0.0609, -0.2806, -0.4480,
         -0.5450, -0.2587, -0.3932, -0.0467,  0.1930, -0.0784, -0.1582, -0.0760,
          0.1110, -0.1461, -0.1338,  0.4218]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.4092, 0.0000, 0.0000, 0.9743, 0.0000, 0.0246, 0.0000, 0.0629, 0.0000,
         0.0000, 0.0000, 0.2805, 0.1764, 0.0338, 0.0000, 0.1114, 0.0753, 0.0130,
         0.0643, 0.4758],
        [0.2024, 0.0000, 0.0000, 0.4911, 0.0000, 0.0904, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.2337, 0.2247, 0.00

nn.Sequential

In [12]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)

input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [13]:
print(logits)

tensor([[ 0.0878,  0.3783, -0.3243,  0.0213, -0.1401,  0.0575,  0.0847,  0.2462,
          0.0480,  0.1704],
        [-0.0137,  0.3466, -0.2431, -0.0605, -0.0655,  0.0578,  0.1134,  0.2501,
          0.0847,  0.1216],
        [ 0.0868,  0.3865, -0.2310, -0.0116, -0.1292, -0.0218,  0.2196,  0.3001,
          0.1204,  0.1398]], grad_fn=<AddmmBackward0>)


In [14]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [15]:
print(pred_probab)

tensor([[0.1008, 0.1348, 0.0668, 0.0944, 0.0803, 0.0978, 0.1005, 0.1181, 0.0969,
         0.1095],
        [0.0918, 0.1316, 0.0730, 0.0876, 0.0872, 0.0986, 0.1042, 0.1195, 0.1013,
         0.1051],
        [0.0985, 0.1329, 0.0717, 0.0892, 0.0793, 0.0883, 0.1125, 0.1219, 0.1018,
         0.1038]], grad_fn=<SoftmaxBackward0>)


Model Parameters

In [16]:
print(f"Model Structure: {model}\n\n")

for name, param in model.named_parameters():
  print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]} \n")

Model Structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[ 0.0113, -0.0109,  0.0210,  ...,  0.0183, -0.0142,  0.0033],
        [ 0.0159, -0.0137,  0.0185,  ...,  0.0166, -0.0192, -0.0055]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([ 0.0283, -0.0230], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[-0.0353, -0.0244,  0.0433,  ..., -0.0319, -0.0389,  0.0335],
        [-0.0180, -0.0346,  0.0391,  ..., -0.0220, -0.0362, -0.0154]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Siz